# Orpheus-3B Sunbird Luganda — vLLM Inference

High-throughput inference notebook for the finetuned Luganda voice [`sunbird/orpheus-3b-tts-salt-lug-0001`](https://huggingface.co/sunbird/orpheus-3b-tts-salt-lug-0001) served by [vLLM](https://docs.vllm.ai). vLLM provides PagedAttention + continuous batching, giving roughly 5–10× faster single-request latency and 10–100× throughput on batched requests vs. plain HuggingFace `model.generate`.

**vs. `Orpheus_3B_Sunbird_Luganda_Inference.ipynb`:** same Orpheus token layout, same SNAC decode path. Only the LM step changes from `transformers` + `unsloth` to `vllm.LLM`. We also showcase batch inference (cell #5) and an OpenAI-compatible deployment server snippet (#7).

**Companion notebook**: `Orpheus_3B_Sunbird_Luganda.ipynb` trains and pushes the merged 16-bit model this notebook serves.

**Why a separate notebook?** vLLM ships its own Torch + transformers and conflicts with unsloth's pinned versions. Use a fresh Python env / GPU instance for vLLM; do not run this notebook in the same environment that ran the training notebook.

## #0 Setup & secrets

vLLM bundles its own torch / transformers, so the install is much smaller than the training notebook. We only add `snac` + `soundfile` for audio decoding and `datasets` for pulling the test split.

In [ ]:
%%capture
!pip install vllm
!pip install snac torchcodec "datasets>=3.4.1,<4.0.0"
# `--ignore-installed blinker` works around vast.ai/Debian images where
# blinker is installed via apt (no RECORD file) and pip refuses to upgrade it.
!pip install --ignore-installed blinker soundfile

In [ ]:
import os
from getpass import getpass

# HF token to pull the finetuned model. Skip the prompt if HF_TOKEN
# is already set in the env.
if not os.environ.get("HF_TOKEN"):
    os.environ["HF_TOKEN"] = getpass("HF_TOKEN: ")

In [ ]:
import locale
from pathlib import Path

import numpy as np
import torch
import soundfile as sf
from datasets import load_dataset, Audio
from snac import SNAC
from transformers import AutoTokenizer
from vllm import LLM, SamplingParams
from IPython.display import display, Audio as AudioDisplay

locale.getpreferredencoding = lambda: "UTF-8"

INFERENCE_OUTPUT_DIR = Path("orpheus-3B/inference_outputs/vllm")
INFERENCE_OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

## #1 Load the finetuned model into vLLM + SNAC decoder

`MODEL_ID` accepts either an HF repo or a local directory (e.g. `orpheus-3B/outputs/orpheus_finetune_16bit`). vLLM wants a **merged** checkpoint — pure LoRA adapters need to be merged first (the training notebook's #5.2 already produces a merged 16-bit copy you can point this at).

**Tuning knobs:**
- `gpu_memory_utilization=0.85` leaves ~15% headroom for SNAC + activation tensors. Drop to `0.70` if you get OOM during model load on a 16 GB GPU.
- `max_model_len=4096` matches the training context length. vLLM pre-allocates KV cache for this; larger = more VRAM consumed even when prompts are short.
- `enforce_eager=False` keeps CUDA graph compilation on (faster decode); set `True` to skip the ~30 s warm-up at startup.

In [ ]:
MODEL_ID = "sunbird/orpheus-3b-tts-salt-lug-0001"
# Or load a local merged checkpoint:
# MODEL_ID = "orpheus-3B/outputs/orpheus_finetune_16bit"

SPEAKER_ID  = "salt_lug_0001"
MAX_MODEL_LEN = 4096

# Orpheus-3B uses vocab_size=156939 (Llama-3 base 128256 + 7*4096 audio
# codebook + 11 special tokens). Unsloth's `push_to_hub_merged` shim can
# add a pad token to the tokenizer at save time, bumping `len(tokenizer)`
# to 156940 and writing that into config.json — while leaving the actual
# embedding weight at [156939, 3072]. transformers tolerates the
# off-by-one; vLLM asserts equality and crashes. Override here so vLLM
# aligns config.vocab_size with the real embedding shape.
llm = LLM(
    model = MODEL_ID,
    dtype = "bfloat16",                # auto-detected on Ampere+
    max_model_len = MAX_MODEL_LEN,
    gpu_memory_utilization = 0.85,
    enforce_eager = False,
    trust_remote_code = False,
    hf_overrides = {"vocab_size": 156939},
)
print(f"vLLM loaded: {MODEL_ID}")

In [ ]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, token=os.environ.get("HF_TOKEN"))
print(f"Tokenizer vocab size: {tokenizer.vocab_size}")

In [ ]:
# SNAC decoder runs on CPU — frees GPU for vLLM's KV cache.
snac_model = SNAC.from_pretrained("hubertsiuzdak/snac_24khz").to("cpu")
print("SNAC 24 kHz decoder loaded (on CPU)")

## #2 Token constants + helpers

Same special-token layout as the training notebook. The audio codebook spans `[128266, 128266 + 7·4096) = [128266, 156938)`; anything outside that range in vLLM's output is treated as garbage and filtered before SNAC decoding.

In [ ]:
# Special tokens (must match #2.6 of the training notebook)
END_OF_TEXT     = 128009
START_OF_SPEECH = 128257
END_OF_SPEECH   = 128258
START_OF_HUMAN  = 128259
END_OF_HUMAN    = 128260
PAD_TOKEN       = 128263
AUDIO_TOKEN_LO  = 128266
AUDIO_TOKEN_HI  = 128266 + 7 * 4096   # exclusive (156938)


def build_prompt_token_ids(text: str, speaker_id: str) -> list[int]:
    """[SOH] + tokenizer(speaker_id: text) + [EOT, EOH]."""
    tagged = f"{speaker_id}: {text}"
    text_ids = tokenizer.encode(tagged, add_special_tokens=True)
    return [START_OF_HUMAN] + text_ids + [END_OF_TEXT, END_OF_HUMAN]


def codes_to_waveform(generated_token_ids: list[int]) -> np.ndarray:
    """vLLM-output token_ids → 24 kHz mono float32 waveform.

    The last SOS marker (128257) appears in the model's output before
    the audio codes start. After cropping past it, we keep only
    audio-range tokens, redistribute through SNAC layers, and decode.
    """
    ids = torch.tensor(generated_token_ids, dtype=torch.int64)

    # Crop on last SOS — discards [SOA, SOS] preamble and any
    # pre-speech meta tokens vLLM may have emitted.
    sos_positions = (ids == START_OF_SPEECH).nonzero(as_tuple=True)[0]
    if len(sos_positions) > 0:
        ids = ids[sos_positions[-1].item() + 1:]
    else:
        print("WARNING: no SOS in vLLM output — will likely produce silence.")

    # Filter to audio-codebook range (strips EOS + any garbage tokens).
    audio_only = ids[(ids >= AUDIO_TOKEN_LO) & (ids < AUDIO_TOKEN_HI)]
    n = (audio_only.size(0) // 7) * 7
    code_list = [t.item() - AUDIO_TOKEN_LO for t in audio_only[:n]]

    layer_1, layer_2, layer_3 = [], [], []
    for i in range(len(code_list) // 7):
        layer_1.append(code_list[7*i])
        layer_2.append(code_list[7*i + 1] - 4096)
        layer_3.append(code_list[7*i + 2] - 2*4096)
        layer_3.append(code_list[7*i + 3] - 3*4096)
        layer_2.append(code_list[7*i + 4] - 4*4096)
        layer_3.append(code_list[7*i + 5] - 5*4096)
        layer_3.append(code_list[7*i + 6] - 6*4096)
    if not layer_1:
        return np.zeros(12000, dtype=np.float32)   # ~0.5s silence fallback

    def clamp(vals):
        return [max(0, min(4095, v)) for v in vals]
    codes = [torch.tensor(clamp(layer_1)).unsqueeze(0),
             torch.tensor(clamp(layer_2)).unsqueeze(0),
             torch.tensor(clamp(layer_3)).unsqueeze(0)]
    waveform = snac_model.decode(codes)
    return waveform.detach().squeeze().to("cpu").numpy().astype(np.float32)

## #3 `synthesize()` — single-prompt entry point

Mirrors the HF-inference notebook's API: `text in → 24 kHz float32 waveform out`. Internally calls `llm.generate(...)` with the speaker-tagged token-IDs prompt and SNAC-decodes the result. For multi-prompt throughput use `synthesize_batch` in #5 instead — calling `synthesize` in a Python loop forfeits vLLM's continuous batching.

In [ ]:
def synthesize(
    text: str,
    speaker_id: str = SPEAKER_ID,
    *,
    max_tokens: int = 1200,
    temperature: float = 0.6,
    top_p: float = 0.95,
    repetition_penalty: float = 1.1,
    seed: int | None = None,
) -> np.ndarray:
    """Synthesize speech for `text`.

    Returns a 1-D float32 numpy array at 24 kHz mono.
    Pass an int `seed` for reproducible output.
    """
    sampling_params = SamplingParams(
        temperature = temperature,
        top_p = top_p,
        repetition_penalty = repetition_penalty,
        max_tokens = max_tokens,
        stop_token_ids = [END_OF_SPEECH],
        skip_special_tokens = False,    # we need raw token_ids
        seed = seed,
    )
    prompt_ids = build_prompt_token_ids(text, speaker_id)
    outputs = llm.generate(
        [{"prompt_token_ids": prompt_ids}],
        sampling_params,
    )
    generated = list(outputs[0].outputs[0].token_ids)
    return codes_to_waveform(generated)

## #4 Free-text inference — try your own Luganda sentence

Edit `prompt` and re-run. The first request pays a one-time CUDA-graph compile cost (~30 s); subsequent calls are near-instant for short utterances.

In [ ]:
prompt = "Mwattu, oli otya?"   # ← put any Luganda sentence here

wav = synthesize(prompt, seed=42)
out_path = INFERENCE_OUTPUT_DIR / "free_text.wav"
sf.write(str(out_path), wav, 24000)
print(f"{prompt}")
print(f"  -> {out_path}  ({len(wav)/24000:.2f}s)")
display(AudioDisplay(wav, rate=24000))

In [ ]:
import itertools, time

PROMPT  = "Mwattu, oli otya? Webale nnyo okwagala Uganda."
SEED    = 42  # fixed seed isolates the effect of (temp, top_p) from RNG noise

GRID = list(itertools.product(
    [0.3, 0.6, 0.8, 1.0],   # temperature axis
    [0.80, 0.90, 0.95],     # top_p axis
))

prompt_ids = build_prompt_token_ids(PROMPT, SPEAKER_ID)
prompts = [{"prompt_token_ids": prompt_ids} for _ in GRID]
sps = [
    SamplingParams(
        temperature=t, top_p=p,
        repetition_penalty=1.1, max_tokens=1200,
        stop_token_ids=[END_OF_SPEECH], skip_special_tokens=False,
        seed=SEED,
    )
    for t, p in GRID
]

t0 = time.perf_counter()
outs = llm.generate(prompts, sps)
print(f"sweep of {len(GRID)} variants in one batch: {time.perf_counter()-t0:.2f}s")

for (t, p), out in zip(GRID, outs):
    wav = codes_to_waveform(list(out.outputs[0].token_ids))
    out_path = INFERENCE_OUTPUT_DIR / f"sweep_t{t:.1f}_p{p:.2f}.wav"
    sf.write(str(out_path), wav, 24000)
    print(f"\nT={t}  top_p={p}  -> {out_path.name}  ({len(wav)/24000:.2f}s)")
    display(AudioDisplay(wav, rate=24000))


## #5 Batch inference — vLLM's main throughput advantage

vLLM's continuous batching schedules many in-flight requests through one shared GPU pass. Passing `N` prompts to a single `llm.generate(...)` call is dramatically faster than calling `synthesize` `N` times in a loop, because the per-request overhead (kernel launches, host↔device sync) is amortized.

Each prompt can specify its own `speaker_id`, so this also demonstrates multi-speaker batching for the multilingual checkpoint.

In [ ]:
def synthesize_batch(
    items: list[dict],
    *,
    max_tokens: int = 1200,
    temperature: float = 0.6,
    top_p: float = 0.95,
    repetition_penalty: float = 1.1,
    seed: int | None = None,
) -> list[np.ndarray]:
    """Synthesize many prompts in one vLLM batch.

    Each `items[i]` is a dict with keys:
      - `text`: the transcript
      - `speaker_id`: optional, defaults to SPEAKER_ID
    Returns a list of 1-D float32 numpy waveforms, same length
    and order as `items`.
    """
    sampling_params = SamplingParams(
        temperature = temperature,
        top_p = top_p,
        repetition_penalty = repetition_penalty,
        max_tokens = max_tokens,
        stop_token_ids = [END_OF_SPEECH],
        skip_special_tokens = False,
        seed = seed,
    )
    prompts = [
        {"prompt_token_ids": build_prompt_token_ids(
            it["text"], it.get("speaker_id", SPEAKER_ID))}
        for it in items
    ]
    outputs = llm.generate(prompts, sampling_params)
    return [codes_to_waveform(list(o.outputs[0].token_ids)) for o in outputs]

In [ ]:
import time

demo_items = [
    {"text": "Mwattu, oli otya?"},
    {"text": "Webale nyo okwagala Uganda."},
    {"text": "Tunaagenda mu Kampala olwa leero."},
    {"text": "Twebale ku kutu."},
]

t0 = time.perf_counter()
wavs = synthesize_batch(demo_items, seed=123)
elapsed = time.perf_counter() - t0
total_audio_sec = sum(len(w) / 24000 for w in wavs)
rtf = elapsed / total_audio_sec if total_audio_sec > 0 else float('inf')
print(f"batch of {len(demo_items)}: {elapsed:.2f}s wall, {total_audio_sec:.2f}s audio,"
      f" real-time-factor = {rtf:.3f} (lower is better, < 1 means faster than realtime)")

for idx, (item, wav) in enumerate(zip(demo_items, wavs)):
    out_path = INFERENCE_OUTPUT_DIR / f"batch_{idx:02d}.wav"
    sf.write(str(out_path), wav, 24000)
    print(f"[{idx}] {item['text']}  ->  {out_path}")
    display(AudioDisplay(wav, rate=24000))

## #6 Held-out test split — A/B vs ground truth

Pulls the first 5 utterances from `Sunbird/tts (lug)` test split, runs them through `synthesize_batch`, and saves each synthesized version next to its ground-truth recording. Same evaluation pattern as the HF-inference notebook — useful as a head-to-head check that vLLM produces equivalent audio quality to `transformers` generate (it should).

In [ ]:
ds_test = load_dataset("Sunbird/tts", "lug", split="test")
ds_test = ds_test.filter(lambda r: r["speaker_id"] == SPEAKER_ID)
ds_test = ds_test.cast_column("audio", Audio(sampling_rate=24000))
print(f"{len(ds_test)} test rows for {SPEAKER_ID}")

n_compare = min(5, len(ds_test))
items = [{"text": ds_test[i]["text"], "speaker_id": SPEAKER_ID}
         for i in range(n_compare)]

t0 = time.perf_counter()
wavs = synthesize_batch(items, seed=42)
print(f"\nvLLM batch of {n_compare}: {time.perf_counter() - t0:.2f}s wall\n")

for i in range(n_compare):
    text = items[i]["text"]
    print(f"[{i}] {text[:120]}")
    synth_path = INFERENCE_OUTPUT_DIR / f"test_{i:02d}_synth.wav"
    truth_path = INFERENCE_OUTPUT_DIR / f"test_{i:02d}_groundtruth.wav"
    sf.write(str(synth_path), wavs[i], 24000)
    sf.write(str(truth_path), ds_test[i]["audio"]["array"], 24000)
    print(f"    synth        -> {synth_path}")
    print(f"    ground truth -> {truth_path}")
    print("    [synth]")
    display(AudioDisplay(wavs[i], rate=24000))
    print("    [ground truth]")
    display(AudioDisplay(ds_test[i]["audio"]["array"], rate=24000))
    print()

## #7 Production deployment with `vllm serve`

vLLM ships with an OpenAI-compatible HTTP server. For TTS we want a custom endpoint that returns `audio/wav` rather than JSON token streams, so the cleanest pattern is to wrap the in-process `LLM` instance behind FastAPI (same as the HF-inference notebook) — but with vLLM's batched generate doing the heavy lifting.

Below is a **runnable** FastAPI server template. Save the snippet to `vllm_tts_server.py`, then `uvicorn vllm_tts_server:app --host 0.0.0.0 --port 8000`. It supports both single-text and batched (multi-text) requests on a single GPU.

**Why not `vllm serve`?** That CLI exposes the OpenAI Chat Completions API and serializes outputs as text/JSON. For TTS you need raw token IDs back so you can run SNAC decoding — the wrapper below is the simplest way.

In [ ]:
VLLM_FASTAPI_SNIPPET = '''
# vllm_tts_server.py — minimal Orpheus-3B TTS server backed by vLLM.
# Run: uvicorn vllm_tts_server:app --host 0.0.0.0 --port 8000
import io
import os
from typing import Optional

import numpy as np
import soundfile as sf
import torch
from fastapi import FastAPI
from fastapi.responses import Response
from pydantic import BaseModel
from snac import SNAC
from transformers import AutoTokenizer
from vllm import LLM, SamplingParams

MODEL_ID  = os.environ.get("ORPHEUS_MODEL_ID", "sunbird/orpheus-3b-tts-salt-lug-0001")
DEFAULT_SPEAKER = os.environ.get("ORPHEUS_SPEAKER", "salt_lug_0001")
MAX_MODEL_LEN = int(os.environ.get("ORPHEUS_MAX_MODEL_LEN", "4096"))

END_OF_TEXT, START_OF_SPEECH, END_OF_SPEECH = 128009, 128257, 128258
START_OF_HUMAN, END_OF_HUMAN = 128259, 128260
AUDIO_TOKEN_LO, AUDIO_TOKEN_HI = 128266, 128266 + 7 * 4096

llm        = LLM(model=MODEL_ID, dtype="bfloat16", max_model_len=MAX_MODEL_LEN,
                 gpu_memory_utilization=0.85,
                 hf_overrides={"vocab_size": 156939})
tokenizer  = AutoTokenizer.from_pretrained(MODEL_ID)
snac_model = SNAC.from_pretrained("hubertsiuzdak/snac_24khz").to("cpu")

def _build_prompt_ids(text, speaker):
    tagged = f"{speaker}: {text}"
    text_ids = tokenizer.encode(tagged, add_special_tokens=True)
    return [START_OF_HUMAN] + text_ids + [END_OF_TEXT, END_OF_HUMAN]

def _codes_to_wav(token_ids):
    ids = torch.tensor(token_ids, dtype=torch.int64)
    sos = (ids == START_OF_SPEECH).nonzero(as_tuple=True)[0]
    if len(sos) > 0: ids = ids[sos[-1].item() + 1:]
    audio = ids[(ids >= AUDIO_TOKEN_LO) & (ids < AUDIO_TOKEN_HI)]
    n = (audio.size(0) // 7) * 7
    cl = [t.item() - AUDIO_TOKEN_LO for t in audio[:n]]
    l1, l2, l3 = [], [], []
    for i in range(len(cl) // 7):
        l1.append(cl[7*i])
        l2.append(cl[7*i+1] - 4096); l3.append(cl[7*i+2] - 2*4096)
        l3.append(cl[7*i+3] - 3*4096); l2.append(cl[7*i+4] - 4*4096)
        l3.append(cl[7*i+5] - 5*4096); l3.append(cl[7*i+6] - 6*4096)
    if not l1: return np.zeros(12000, dtype=np.float32)
    cb = lambda v: [max(0, min(4095, x)) for x in v]
    codes = [torch.tensor(cb(l1)).unsqueeze(0), torch.tensor(cb(l2)).unsqueeze(0),
             torch.tensor(cb(l3)).unsqueeze(0)]
    return snac_model.decode(codes).detach().squeeze().cpu().numpy().astype(np.float32)

app = FastAPI()

class TTSReq(BaseModel):
    text: str
    speaker_id: str = DEFAULT_SPEAKER
    seed: Optional[int] = None
    temperature: float = 0.6
    top_p: float = 0.95
    repetition_penalty: float = 1.1
    max_tokens: int = 1200

@app.post("/tts", responses={200: {"content": {"audio/wav": {}}}})
def tts(req: TTSReq):
    sp = SamplingParams(
        temperature=req.temperature, top_p=req.top_p,
        repetition_penalty=req.repetition_penalty,
        max_tokens=req.max_tokens, stop_token_ids=[END_OF_SPEECH],
        skip_special_tokens=False, seed=req.seed,
    )
    pids = _build_prompt_ids(req.text, req.speaker_id)
    out = llm.generate([{"prompt_token_ids": pids}], sp)
    wav = _codes_to_wav(list(out[0].outputs[0].token_ids))
    buf = io.BytesIO()
    sf.write(buf, wav, 24000, format="WAV", subtype="PCM_16")
    return Response(content=buf.getvalue(), media_type="audio/wav")

class TTSBatchReq(BaseModel):
    items: list[TTSReq]

@app.post("/tts/batch")
def tts_batch(req: TTSBatchReq):
    sp = SamplingParams(
        temperature=req.items[0].temperature, top_p=req.items[0].top_p,
        repetition_penalty=req.items[0].repetition_penalty,
        max_tokens=req.items[0].max_tokens,
        stop_token_ids=[END_OF_SPEECH], skip_special_tokens=False,
    )
    prompts = [{"prompt_token_ids": _build_prompt_ids(it.text, it.speaker_id)} for it in req.items]
    outs = llm.generate(prompts, sp)
    wavs = [_codes_to_wav(list(o.outputs[0].token_ids)) for o in outs]
    # Multipart return omitted for brevity — typically zip the wavs or stream them.
    return {"sample_rates": [24000] * len(wavs), "durations_sec": [len(w)/24000 for w in wavs]}
'''
print(VLLM_FASTAPI_SNIPPET)